# Part 3 - 02 Data preprocessing: cleaning, split and YOLO dataset

Input: `data/raw_bbox` (never modified). Output: `data/yolo` with `images/` and `labels/` split into `train`, `valid`, `test`, plus `data.yaml`.

**Steps**
1. Inventory of every file (extension, readability, label, hash)
2. Cleaning decisions, each one counted: bad extensions, corrupted images, orphans, invalid or empty annotations, class inconsistencies, duplicates
3. Stratified 70/15/15 split with a fixed seed, no leakage
4. Export in YOLO layout and `data.yaml`
5. Verification: counts, pairs, valid labels, visual check

**Decisions made in this notebook**
- Empty annotation: excluded. "No Tumor" images carry a real box of class 2, so an empty file is ambiguous (healthy brain or forgotten label) and we do not guess.
- Image whose boxes do not include its folder class: excluded (`MIXED_POLICY`). Images with an extra box of another class are kept by default because they may be real multi-tumor images; change the policy after looking at the examples.
- Byte-identical duplicates: one copy kept; copies present under different classes are all removed.
- The original Train/Valid folders are **pooled and re-split**, because duplicates across them would leak between sets.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from neurovision.yolo_data import (CLASS_NAMES, SEED, apply_cleaning, build_inventory, draw_boxes,
                                   find_orphan_labels, set_seed, stratified_split,
                                   verify_yolo_dataset, write_data_yaml, write_yolo_dataset)

set_seed(SEED)

ROOT = Path("../data/raw_bbox")     # raw data, read only
OUT = Path("../data/yolo")          # clean dataset, rebuilt at each run
REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)

MIXED_POLICY = "drop_inconsistent"  # "keep" | "drop_inconsistent" | "drop_all_mixed"
RATIOS = (0.70, 0.15, 0.15)         # train / valid / test

## 1. Inventory (reads every image once: takes a minute)

In [ ]:
df = build_inventory(ROOT)
df["folder_name"] = df["folder_class"].map(lambda i: CLASS_NAMES[int(i)] if pd.notna(i) else "unknown")
orphan_labels = find_orphan_labels(ROOT, df)
print(f"{len(df)} files inventoried")

## 2. Pairing, extensions and corrupted images

In [ ]:
step1 = pd.Series({
    "non-image files (bad extension)": int((~df.ext_ok).sum()),
    "corrupted / unreadable images": int((df.readable == False).sum()),   # noqa: E712
    "images without a label file": int((df.label_status == "missing").sum()),
    "label files without an image": len(orphan_labels),
})
display(step1.to_frame("count"))
display(df.loc[~df.ext_ok, "rel"].head(10).to_frame())
display(pd.Series([p.name for p in orphan_labels[:10]], name="orphan labels (first 10)").to_frame())

## 3. Annotation validation (5 values, valid integer class, coordinates in [0, 1], positive size)

In [ ]:
print(df["label_status"].value_counts(dropna=False).to_dict())

invalid = df[df.label_status == "invalid"]
display(invalid["label_reason"].str.replace(r"line \d+: ", "", regex=True).value_counts().to_frame("count"))
display(invalid[["rel", "label_reason"]].head(10))

## 4. Empty annotations

"No Tumor" images are annotated with a box of class 2, so an **empty** `.txt` is not a normal healthy brain here: it is either a forgotten label or an unexplained case. We exclude them rather than guess.

In [ ]:
empty = df[df.label_status == "empty"]
print("Empty annotation files:", len(empty))
display(empty.groupby("folder_name").size().to_frame("empty annotations"))

## 5. Class consistency between the folder and the boxes

In [ ]:
rows = {}
for policy in ("keep", "drop_inconsistent", "drop_all_mixed"):
    rows[policy] = int((apply_cleaning(df, policy).excluded == "inconsistent_class").sum())
display(pd.Series(rows, name="images excluded as inconsistent").to_frame())
print("Selected policy:", MIXED_POLICY)

In [ ]:
# Examples excluded by the strictest policy: look at them to validate the decision
strict = apply_cleaning(df, "drop_all_mixed")
flagged = strict[strict.excluded == "inconsistent_class"]
if len(flagged):
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for ax, (_, r) in zip(axes.ravel(), flagged.sample(min(6, len(flagged)), random_state=SEED).iterrows()):
        ax.imshow(draw_boxes(r.path, r.label_path))
        ax.set_title(f"{r.folder_name} folder, ids={sorted(set(r.classes))}", fontsize=9)
    for ax in axes.ravel():
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Duplicates (identical file content)

In [ ]:
hashed = df[df.hash.notna()]
groups = hashed.groupby("hash")
group_size = groups.size()
dup_hashes = group_size[group_size > 1].index

print(f"{len(dup_hashes)} duplicate groups, {int((group_size[dup_hashes] - 1).sum())} redundant files")
print("Groups spanning different classes:", int((groups["folder_class"].nunique()[dup_hashes] > 1).sum()))
print("Groups spanning several original splits (leakage risk in the raw data):",
      int((groups["split"].nunique()[dup_hashes] > 1).sum()))

## 7. Apply the cleaning (raw data untouched) and report

In [ ]:
clean = apply_cleaning(df, MIXED_POLICY)
kept = clean[clean.excluded == ""]

reasons = clean.loc[clean.excluded != "", "excluded"].value_counts()
display(reasons.to_frame("excluded files"))
print(f"{len(clean)} files inventoried -> {len(kept)} images kept")

before = df.loc[df.ext_ok, "folder_name"].value_counts()
after = kept["folder_name"].value_counts()
comparison = pd.DataFrame({"before": before, "after": after}).fillna(0).astype(int)
display(comparison)
comparison.plot.bar(rot=0, title="Images per class: before and after cleaning")
plt.tight_layout()
plt.savefig(REPORTS / "part3_cleaning_effect.png", dpi=150)
plt.show()

In [ ]:
report = {
    "seed": SEED, "mixed_policy": MIXED_POLICY,
    "inventoried_files": int(len(clean)), "kept_images": int(len(kept)),
    "orphan_label_files": len(orphan_labels),
    "excluded_by_reason": {k: int(v) for k, v in reasons.items()},
    "images_per_class_after": {k: int(v) for k, v in after.items()},
}
(REPORTS / "part3_cleaning_report.json").write_text(json.dumps(report, indent=2))

excluded = clean.loc[clean.excluded != "", ["rel", "excluded"]]
(REPORTS / "part3_excluded_files.json").write_text(
    json.dumps(dict(zip(excluded.rel, excluded.excluded)), indent=2))
print("Reports saved in", REPORTS)

## 8. Stratified split (same seed, same proportions of each class, before any augmentation)

In [ ]:
splits = stratified_split(clean, seed=SEED, ratios=RATIOS)

print({name: len(part) for name, part in splits.items()})
proportions = pd.DataFrame({name: part["folder_name"].value_counts(normalize=True)
                            for name, part in splits.items()}).round(3)
display(proportions)

### No leakage: no shared file and no shared content between the sets

In [ ]:
paths = {name: set(part["path"]) for name, part in splits.items()}
hashes = {name: set(part["hash"]) for name, part in splits.items()}

for a, b in (("train", "valid"), ("train", "test"), ("valid", "test")):
    shared_files, shared_content = len(paths[a] & paths[b]), len(hashes[a] & hashes[b])
    print(f"{a} / {b}: shared files = {shared_files}, shared contents = {shared_content}")
    assert shared_files == 0 and shared_content == 0

## 9. Export in YOLO layout and `data.yaml`

In [ ]:
# overwrite=True deletes and rebuilds data/yolo only (never the raw folder)
write_yolo_dataset(splits, OUT, overwrite=True)
yaml_path = write_data_yaml(OUT)
print(yaml_path.read_text())

## 10. Verification: counts, pairs, valid labels

In [ ]:
summary = verify_yolo_dataset(OUT)
display(summary)

assert (summary["images"] == summary["labels"]).all()
assert (summary[["images_without_label", "labels_without_image", "invalid_labels"]] == 0).all().all()
for name, part in splits.items():
    assert summary.loc[name, "images"] == len(part)
print("All checks passed: every image has its label, every label is valid, counts match the split.")

summary.reset_index().to_json(REPORTS / "part3_split_summary.json", orient="records", indent=2)

In [ ]:
# Visual check on the exported files (boxes must still sit on the tumors)
files = sorted((OUT / "images" / "train").glob("*"))
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, p in zip(axes.ravel(), files[:: max(1, len(files) // 6)][:6]):
    ax.imshow(draw_boxes(p))
    ax.set_title(p.name[:35], fontsize=7)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Summary

- Raw data untouched; every exclusion is counted in `reports/part3_cleaning_report.json` and listed in `reports/part3_excluded_files.json`.
- `data/yolo/` is ready for Ultralytics through `data/yolo/data.yaml`.
- Augmentation is **not** applied here: in YOLOv8 it is configured as training arguments (rotation, scale, translation, brightness, no horizontal flip, Mosaic tuning), so it only touches the training set. That belongs to the training notebook.

Next: `part3_03_training.ipynb` (1-epoch sanity run, then full training).